### Step 4a: Multi-Label Encoding & Patient-Level Split

#### Cell 1: Load curated data & re-apply multi-label encoding

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/metadata_curated.csv')

disease_list = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
                 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
                 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']

for disease in disease_list:
    df[disease] = df['Finding Labels'].apply(lambda x: 1 if disease in x else 0)

print(df.shape)
print("Unique patients:", df['Patient ID'].nunique())

(112120, 27)
Unique patients: 30805


#### Cell 2: Patient-level train/val/test split

In [2]:
from sklearn.model_selection import train_test_split

unique_patients = df['Patient ID'].unique()

train_patients, temp_patients = train_test_split(unique_patients, test_size=0.3, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.5, random_state=42)

train_df = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
test_df = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

print(f"Train: {len(train_df)} images, {train_df['Patient ID'].nunique()} patients")
print(f"Val:   {len(val_df)} images, {val_df['Patient ID'].nunique()} patients")
print(f"Test:  {len(test_df)} images, {test_df['Patient ID'].nunique()} patients")

Train: 78566 images, 21563 patients
Val:   17063 images, 4621 patients
Test:  16491 images, 4621 patients


#### Cell 3: Verify zero patient leakage (critical check)

In [3]:
train_ids = set(train_df['Patient ID'])
val_ids = set(val_df['Patient ID'])
test_ids = set(test_df['Patient ID'])

print("Train-Val overlap:", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap:", len(val_ids & test_ids))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


#### Cell 4: Save the splits

In [4]:
train_df.to_csv('../data/processed/train_split.csv', index=False)
val_df.to_csv('../data/processed/val_split.csv', index=False)
test_df.to_csv('../data/processed/test_split.csv', index=False)

print("Saved all three splits to data/processed/")

Saved all three splits to data/processed/


#### Step 4b — src/dataset.py (PyTorch Dataset class)

In [6]:
import sys
sys.path.append('../src')
from dataset import build_image_path_map, ChestXrayDataset

image_map = build_image_path_map('../data/raw', cache_path='../data/processed/image_path_map.json')
print("Total images mapped:", len(image_map))

train_dataset = ChestXrayDataset('../data/processed/train_split.csv', image_map, split='train')
print("Train dataset size:", len(train_dataset))

img, label = train_dataset[0]
print("Image tensor shape:", img.shape)
print("Label vector:", label)

Total images mapped: 112120
Train dataset size: 78566
Image tensor shape: torch.Size([3, 224, 224])
Label vector: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


#### Step 5a: Create the Filtered Subset

In [2]:
import pandas as pd

train_df = pd.read_csv('../data/processed/train_split.csv')
val_df = pd.read_csv('../data/processed/val_split.csv')
test_df = pd.read_csv('../data/processed/test_split.csv')

disease_list = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
                 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
                 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']

print(train_df.shape, val_df.shape, test_df.shape)

(78566, 27) (17063, 27) (16491, 27)


#### Cell B — create and save the subsets:

In [3]:
train_subset = train_df.sample(n=15000, random_state=42).reset_index(drop=True)
val_subset = val_df.sample(n=3000, random_state=42).reset_index(drop=True)
test_subset = test_df.sample(n=3000, random_state=42).reset_index(drop=True)

train_subset.to_csv('../data/processed/train_subset.csv', index=False)
val_subset.to_csv('../data/processed/val_subset.csv', index=False)
test_subset.to_csv('../data/processed/test_subset.csv', index=False)

print("Subset disease counts (train):")
print(train_subset[disease_list].sum().sort_values(ascending=False))

Subset disease counts (train):
Infiltration          2619
Effusion              1793
Atelectasis           1568
Nodule                 896
Mass                   805
Pneumothorax           716
Consolidation          580
Pleural_Thickening     461
Cardiomegaly           366
Emphysema              313
Edema                  301
Fibrosis               227
Pneumonia              196
Hernia                  29
dtype: int64


In [4]:
ls -la ../data/processed/

total 30040
drwxrwxr-x 2 isha_sas_ai isha_sas_ai    4096 Aug  4 17:01 ./
drwxrwxr-x 4 isha_sas_ai isha_sas_ai    4096 Aug  3 05:15 ../
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 7848400 Aug  4 07:41 image_path_map.json
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 8758035 Aug  4 05:01 metadata_curated.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 1751065 Aug  4 07:05 test_split.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai  318730 Aug  4 17:01 test_subset.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 8335495 Aug  4 07:05 train_split.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 1591096 Aug  4 17:01 train_subset.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai 1811618 Aug  4 07:05 val_split.csv
-rw-rw-r-- 1 isha_sas_ai isha_sas_ai  318712 Aug  4 17:01 val_subset.csv
